# Zava Retail Agent: SFT, RFT, deployment, and comparison

This capstone trains supervised and reinforcement fine-tuned retail agents, inspects service training metrics, verifies customer-provisioned deployments, and compares base and fine-tuned behavior on fixed held-out scenarios.

Training and inference can incur charges. Run only the stages you intend to complete.

> Actual results may vary by model version, data, configuration, region availability, and service conditions.

## 1. Configure the customer-owned resources

Install the repository shared dependencies, copy this cookbook's `.env.template` to a module-local `.env`, and configure your own project endpoints, isolated tool server, account names, and unique deployment names. Deployment readiness checks are read-only; the notebook never creates, replaces, or deletes deployments.

This capstone does not ship a retail server. Supply an approved external implementation of all 17 operations in the [retail API contract](../configs/retail-openapi-with-policy.json) and follow the [external server requirements](../SERVER_CONTRACT.md). The nearby catalog sample only implements `search_catalog`; it is not a retail backend. The preserved data alone is not a complete return-policy engine.

RFT training uses mutating tools. Point `FOUNDRY_TOOLS_SERVER_URL` to an approved HTTPS server backed by an isolated disposable copy of the retail fixture, never production, and set `FOUNDRY_TOOL_SERVER_IS_SANDBOX=true` only after confirming isolation. Inject the same high-entropy `FOUNDRY_TOOL_AUTH_TOKEN` into the server and notebook environment. Both training headers and local preflight/comparison use `Authorization: Bearer ...`; a live stage cannot fall back to anonymous access. Do not display the live recipe or enable HTTP/SDK wire-debug logging. Comparison permits only the read-only allowlist and never executes a blocked call.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import importlib.util
import io
import json
import math
import os
import re
import sys
import time
import uuid
from pathlib import Path
from urllib.parse import quote, urlparse
from dotenv import load_dotenv

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'manifest.json').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'data' / 'manifest.json').is_file():
    raise RuntimeError('Run from the cookbook directory or notebooks/.')
sys.path.insert(0, str(ROOT.parents[1]))
sys.path.insert(0, str(ROOT / 'src'))
from shared.retry import RetryPolicy, retry_call
from shared.config import load_foundry_config
from shared.foundry_operations import wait_for_file, monitor_fine_tuning_job
from retail_tools import (
    build_remote_tools, call_read_only_tool, load_tool_headers,
    preflight_retail_tools, validate_local_tool_contract,
    )
from retail_curation import prepare_active_datasets, validated_prepared_paths

load_dotenv(ROOT / '.env', override=False)

def setting(name, default=''):
    return os.getenv(name, '').strip() or default

def env_flag(name):
    value = setting(name, 'false').casefold()
    if value not in {'true', 'false'}:
        raise ValueError(f'{name} must be true or false (blank means false).')
    return value == 'true'

def required(name):
    value = setting(name)
    if not value:
        raise ValueError(f'Set {name} explicitly before running this stage.')
    return value

def positive_integer(name, default=None):
    value = setting(name, str(default) if default is not None else '')
    if not value.isdecimal() or int(value) < 1:
        raise ValueError(f'{name} must be an explicit positive integer.')
    return int(value)

def nonnegative_number(name, default=0):
    value = setting(name, str(default))
    try:
        parsed = float(value)
    except ValueError as exc:
        raise ValueError(f'{name} must be a non-negative number.') from exc
    if parsed < 0:
        raise ValueError(f'{name} must be a non-negative number.')
    return parsed

def https_url(value, label):
    parsed = urlparse(value)
    if parsed.scheme != 'https' or not parsed.hostname or parsed.username or parsed.password or parsed.query or parsed.fragment:
        raise ValueError(f'{label} must be an HTTPS URL without credentials, query, or fragment.')
    return value.rstrip('/')

def project_endpoint(name):
    return load_foundry_config(environ={'FOUNDRY_PROJECT_ENDPOINT': required(name)}).project_endpoint

_paid_warning_emitted = False

def warn_paid_once():
    global _paid_warning_emitted
    if not _paid_warning_emitted:
        print('This workflow uses paid cloud resources and may incur charges.')
        _paid_warning_emitted = True

SFT_MODEL = 'gpt-4.1-mini-2025-04-14'
RFT_MODEL = 'o4-mini-2025-04-16'
TRAINING_TYPE = 'globalStandard'
SFT_ENDPOINT = setting('FOUNDRY_SFT_PROJECT_ENDPOINT')
RFT_ENDPOINT = setting('FOUNDRY_RFT_PROJECT_ENDPOINT')
TOOL_SERVER_URL = setting('FOUNDRY_TOOLS_SERVER_URL').rstrip('/')
TOOL_HEADERS = load_tool_headers()
POLL_SECONDS = positive_integer('FOUNDRY_POLL_INTERVAL_SECONDS', 60)
WORKERS = positive_integer('FOUNDRY_COMPARISON_WORKERS', 1)
COMPARISON_PACING_SECONDS = nonnegative_number('FOUNDRY_COMPARISON_PACING_SECONDS', 1)
COMPARISON_RETRY_POLICY = RetryPolicy(max_attempts=12, initial_delay_seconds=10,
    max_delay_seconds=120, multiplier=2, jitter_ratio=0.1, timeout_seconds=900)
DATA, CONFIGS, OUTPUTS = ROOT / 'data', ROOT / 'configs', ROOT / 'outputs'
STATE_PATH = OUTPUTS / 'notebook-state.json'

## 2. Verify raw provenance and prepare validated active copies

All committed datasets are immutable raw provenance, not trusted active inputs. Offline preparation verifies every preserved input against its LF-canonical byte count, SHA-256 hash, and record count, then creates only five active SFT/RFT training, validation, and next-tool copies under ignored `outputs/prepared-data/`. Only reviewed rows are curated: existing successful authentication blocks move before catalog access; impossible payment mutations become refusals. No observed tool result is invented, no source row is overwritten, and synthetic source splits are never promoted to active data.

Every prepared row must pass authentication-first and observed, customer-owned payment checks. The generated LF manifest records actual raw/canonical source hashes, output hashes/counts, exact changed-row evidence, curation decisions, and all findings from all seven raw datasets, including the preserved synthetic originals. Those findings are reported, not waived as passing. Unchanged active rows retain their serialization apart from physical CRLF-to-LF normalization. Repeating preparation reuses byte-identical outputs; source drift, unreviewed unsafe active rows, links, traversal, or conflicting existing outputs fail explicitly without overwriting them.

Uploads and comparisons use only these validated prepared paths and actual output hashes, rechecked immediately before use. The historical manifest remains unchanged. The active API retains the original 15 operations plus the two RFT-only operations and bearer security; offline contract checks cover source references and prepared rows. This bounded trace audit does not qualify the external return-policy engine.

In [ ]:
prepared_paths, preparation_report = prepare_active_datasets(ROOT)
retail_operations = validate_local_tool_contract(ROOT, dataset_paths=prepared_paths)
validated_inputs = {
    path.relative_to(ROOT).as_posix(): preparation_report['files'][f'data/{name}']['output']['raw_sha256']
    for name, path in prepared_paths.items()
}
validated_inputs.update({
    relative: entry['sha256'] for relative, entry in preparation_report['source_inputs'].items()
    if not relative.startswith('data/')
})
validated_inputs['outputs/prepared-data/manifest.json'] = hashlib.sha256(
    (OUTPUTS / 'prepared-data' / 'manifest.json').read_bytes()).hexdigest()

def checked_prepared_path(name):
    current = validated_prepared_paths(ROOT, preparation_report)
    if current != prepared_paths or name not in current:
        raise RuntimeError('Prepared paths changed; rerun offline preparation before any service stage.')
    path = current[name]
    if validated_inputs.get(path.relative_to(ROOT).as_posix()) != hashlib.sha256(path.read_bytes()).hexdigest():
        raise RuntimeError('Prepared input hash changed; reconcile saved state before continuing.')
    return path

def checked_prepared_bytes(name):
    content = checked_prepared_path(name).read_bytes()
    expected = preparation_report['files'][f'data/{name}']['output']['raw_sha256']
    if hashlib.sha256(content).hexdigest() != expected:
        raise RuntimeError('Prepared input changed while being read; no upload/evaluation may use it.')
    return content

print('Prepared five validated active copies; all seven raw datasets remain immutable.')
print({'raw_source_findings': {
    relative: len(entry['source_diagnostics']) for relative, entry in preparation_report['files'].items()
}, 'prepared_rows': {name: preparation_report['files'][f'data/{name}']['output']['records']
    for name in prepared_paths}})

## 3. Preserve the original training recipes

SFT: batch size 1, learning-rate multiplier 2, three epochs. RFT: original o3-mini policy grader and remote tools, one epoch. The preserved configuration has empty headers by design; the local builder injects the environment-backed bearer header in memory without rewriting inputs. The placeholder tool URL remains local until an external server is configured. Never display `tools` or `rft_method`, or persist them without redaction.

In [ ]:
policy_grader = json.loads((CONFIGS / 'rft-policy-grader.json').read_text(encoding='utf-8'))
tools = build_remote_tools(CONFIGS / 'rft-tools.json', TOOL_SERVER_URL, TOOL_HEADERS)
sft_method = {'type': 'supervised', 'supervised': {'hyperparameters': {
    'batch_size': 1, 'learning_rate_multiplier': 2, 'n_epochs': 3,
}}}
rft_method = {'type': 'reinforcement', 'reinforcement': {
    'hyperparameters': {'eval_interval': 1, 'eval_samples': 2, 'compute_multiplier': 1,
        'reasoning_effort': 'medium', 'n_epochs': 1, 'batch_size': 10, 'learning_rate_multiplier': 1},
    'grader': policy_grader, 'tools': tools,
}}
# Never display tools/rft_method: their live headers contain the bearer secret.
print({'sft_model': SFT_MODEL, 'rft_model': RFT_MODEL, 'grader': policy_grader['model']})

## 4. Authenticate only for enabled stages

Use `az login` before enabling a service stage. All model operations use the OpenAI clients returned by `AIProjectClient.get_openai_client()`. Automatic SDK retries are disabled for writes; shared bounded retries are used only for reads/inference.

In [ ]:
resources_to_close = []
SFT_ENDPOINT = project_endpoint('FOUNDRY_SFT_PROJECT_ENDPOINT')
RFT_ENDPOINT = project_endpoint('FOUNDRY_RFT_PROJECT_ENDPOINT')
TOOL_SERVER_URL = https_url(required('FOUNDRY_TOOLS_SERVER_URL'), 'Tool server')
if not env_flag('FOUNDRY_TOOL_SERVER_IS_SANDBOX'):
    raise ValueError('RFT requires an explicitly acknowledged isolated sandbox tool server.')
TOOL_HEADERS = load_tool_headers(required=True)
if rft_method['reinforcement']['tools'] != build_remote_tools(CONFIGS / 'rft-tools.json', TOOL_SERVER_URL, TOOL_HEADERS):
    raise RuntimeError('Tool settings changed; rerun the local recipe cell before any service stage.')
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
credential = DefaultAzureCredential()
resources_to_close.append(credential)
for label, endpoint in [('sft', SFT_ENDPOINT), ('rft', RFT_ENDPOINT)]:
    project = AIProjectClient(endpoint=endpoint, credential=credential, allow_preview=True)
    resources_to_close.append(project)
    client = project.get_openai_client(max_retries=0)
    resources_to_close.append(client)
    if label == 'sft':
        sft_client = client
    else:
        rft_client = client


## 5. Optional upload, submission, and monitoring

State is bound to customer endpoints, exact inputs, models, redacted methods, and a credential fingerprint. Live bearer headers are never serialized. Keep state to resume. Older state, credential rotation, or changed configuration fails closed and requires explicit reconciliation; do not delete state to bypass an uncertain paid request. A pending upload/submission after an interrupted request requires reconciliation with the project before editing state. The visible SDK submission preserves `trainingType=globalStandard`; SDK write retries are disabled and pending state is saved before the POST. Monitoring uses the shared helper, has a 24-hour timeout, and does not cancel jobs. Errors from secret-bearing submission/monitoring paths are reported without underlying request or service text.

In [ ]:
def redact_runtime_value(value):
    secret_values = set()
    for tool in rft_method.get('reinforcement', {}).get('tools', []):
        for header in tool.get('headers', {}).values():
            if isinstance(header, str) and header:
                secret_values.add(header)
                if header.casefold().startswith('bearer '):
                    secret_values.add(header[7:])
    secrets = sorted(secret_values | {json.dumps(secret)[1:-1] for secret in secret_values}, key=len, reverse=True)
    def text(item):
        for secret in secrets:
            item = item.replace(secret, '<redacted>')
        return item
    def scrub(item):
        if isinstance(item, dict):
            cleaned = {}
            for key, child in item.items():
                name = str(key)
                if name.casefold() == 'headers':
                    cleaned[text(name)] = {text(str(header)): '<redacted>' for header in child} if isinstance(child, dict) else '<redacted>'
                elif name.casefold() in {'authorization', 'x-functions-key', 'api-key', 'api_key', 'access_token', 'token'}:
                    cleaned[text(name)] = '<redacted>'
                else:
                    cleaned[text(name)] = scrub(child)
            return cleaned
        if isinstance(item, (list, tuple)):
            return [scrub(child) for child in item]
        return text(item) if isinstance(item, str) else item
    return scrub(value)

def tool_auth_fingerprint():
    import hashlib
    headers = [tool.get('headers', {}) for tool in rft_method.get('reinforcement', {}).get('tools', [])]
    return hashlib.sha256(json.dumps(headers, sort_keys=True, separators=(',', ':')).encode('utf-8')).hexdigest()

def save_state():
    OUTPUTS.mkdir(exist_ok=True)
    temporary = STATE_PATH.with_suffix('.tmp')
    temporary.write_text(json.dumps(redact_runtime_value(state), indent=2, sort_keys=True), encoding='utf-8')
    temporary.replace(STATE_PATH)

context = {'schema_version': 3, 'endpoints': [SFT_ENDPOINT, RFT_ENDPOINT], 'inputs': validated_inputs,
    'models': [SFT_MODEL, RFT_MODEL], 'training_type': TRAINING_TYPE,
    'methods': [redact_runtime_value(sft_method), redact_runtime_value(rft_method)],
    'tool_auth_fingerprint': tool_auth_fingerprint()}
state = None
if STATE_PATH.is_file():
    state = json.loads(STATE_PATH.read_text(encoding='utf-8'))
    if state.get('context') != context:
        raise RuntimeError('Runtime state belongs to different inputs/configuration or credentials. Reconcile it explicitly; do not clear pending submissions.')
else:
    state = {'context': context, 'run_token': uuid.uuid4().hex[:16], 'files': {}, 'jobs': {}}
    save_state()

def upload_once(client, label, name):
    path = checked_prepared_path(name)
    payload = checked_prepared_bytes(name)
    digest = hashlib.sha256(payload).hexdigest()
    record = state['files'].get(label)
    if record is not None and record.get('input_sha256') != digest:
        raise RuntimeError('Saved upload belongs to different prepared bytes; reconcile it explicitly.')
    if record is None:
        record = state['files'][label] = {'pending': True,
            'filename': f"retail-{state['run_token']}-{path.name}",
            'input_path': path.relative_to(ROOT).as_posix(), 'input_sha256': digest}
        save_state()
        with io.BytesIO(payload) as stream:
            uploaded = client.files.create(file=(record['filename'], stream, 'application/jsonl'), purpose='fine-tune')
        record = state['files'][label] = {
            'id': uploaded.id, 'input_path': path.relative_to(ROOT).as_posix(), 'input_sha256': digest}
        save_state()
    if not record.get('id'):
        raise RuntimeError(f'Uncertain upload for {label}; reconcile the recorded filename before retrying.')
    wait_for_file(client, record['id'], poll_interval_seconds=10, timeout_seconds=600)
    return record['id']

def submit_once(client, label, model, method):
    record = state['jobs'].get(label)
    if record is None:
        suffix = f"retail-{label}-{state['run_token']}"
        state['jobs'][label] = {'pending': True, 'suffix': suffix}
        save_state()
        failed = False
        try:
            job = client.fine_tuning.jobs.create(model=model,
                training_file=state['files'][f'{label}_train']['id'],
                validation_file=state['files'][f'{label}_validation']['id'],
                suffix=suffix, method=method, extra_body={'trainingType': TRAINING_TYPE})
        except Exception:
            failed = True
        if failed:
            raise RuntimeError('Retail job POST failed or its outcome is uncertain. Reconcile the saved pending request; do not resubmit automatically.') from None
        record = state['jobs'][label] = {'id': job.id, 'suffix': suffix}
        save_state()
    if not record.get('id'):
        raise RuntimeError(f'Uncertain submission for {label}; reconcile the recorded suffix before retrying.')
    return record['id']

### Upload, train, and monitor

Execute this cell to warn once about paid resources, check the authenticated external tool envelope with read-only fixture calls, then upload inputs and submit or resume both jobs without another budget gate. Preflight calls `find_user_id_by_email`, `list_user_orders`, `get_order_details`, and `policy_verify_return`; it never changes the database and does not prove policy correctness. Follow the external-server policy qualification requirements before using it. For comparisons using saved successful jobs, skip this cell and the metric-download cell, then run deployment readiness and comparison cells.

In [ ]:
checked_prepared_path('rft-train.jsonl')
warn_paid_once()
# Authenticated read-only dispatch check, never a policy-engine implementation test.
preflight_retail_tools(ROOT, TOOL_SERVER_URL, dataset_paths=prepared_paths, headers=TOOL_HEADERS,
    sandbox=env_flag('FOUNDRY_TOOL_SERVER_IS_SANDBOX'))
for label, client in [('sft', sft_client), ('rft', rft_client)]:
    for split in ['train', 'validation']:
        upload_once(client, f'{label}_{split}', f'{label}-{split}.jsonl')
submit_once(sft_client, 'sft', SFT_MODEL, sft_method)
submit_once(rft_client, 'rft', RFT_MODEL, rft_method)
for label, client in [('sft', sft_client), ('rft', rft_client)]:
    monitor_failed = False
    try:
        job = monitor_fine_tuning_job(client, state['jobs'][label]['id'], poll_interval_seconds=POLL_SECONDS)
    except Exception:
        monitor_failed = True
    if monitor_failed:
        raise RuntimeError('Retail monitoring failed. Saved job IDs are retained; inspect the job in your project. No job was cancelled or resubmitted.') from None
    state['jobs'][label].update(status=job.status, fine_tuned_model=job.fine_tuned_model, result_files=job.result_files or [])
    save_state()

## 6. Optional service metrics

Download only service-produced CSVs; absence of expected metrics is an error, not an invented curve.

In [ ]:
def download_result(client, label):
    record = state['jobs'][label]
    if not record['result_files']:
        raise RuntimeError(f'{label} succeeded without a service result file.')
    content = retry_call(lambda: client.files.content(record['result_files'][0]).read(), operation_name='download metrics')
    (OUTPUTS / f'{label}-results.csv').write_bytes(content)
    return list(csv.DictReader(io.StringIO(content.decode('utf-8-sig'))))

def metric_curve(rows, field):
    points = [(int(row['step']), float(row[field])) for row in rows if row.get(field)]
    if not points or not all(math.isfinite(value) for _, value in points):
        raise RuntimeError(f'Missing or non-finite service metric: {field}')
    return points

import matplotlib.pyplot as plt
sft_rows = download_result(sft_client, 'sft')
rft_rows = download_result(rft_client, 'rft')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(*zip(*metric_curve(sft_rows, 'valid_loss')), marker='o')
axes[0].set(title='SFT service validation loss', xlabel='Step', ylabel='Loss')
for field, label in [('train_mean_reward', 'Training'), ('full_valid_mean_reward', 'Validation')]:
    axes[1].plot(*zip(*metric_curve(rft_rows, field)), marker='o', label=label)
axes[1].set(title='RFT service reward', xlabel='Step', ylabel='Reward')
axes[1].legend()
plt.tight_layout()


## 7. Provision through your approved process, then verify readiness

After successful training, provision GlobalStandard deployments through your approved Foundry portal or management workflow using the saved fine-tuned artifacts in `outputs/notebook-state.json`. Choose four distinct unique deployment names and appropriate capacity in that workflow. This notebook only discovers and verifies existing deployments: it never creates, updates, or deletes resources, and makes no claim about ARM conditional-create support. Missing resources, authorization errors, and model/version/format/SKU mismatches fail explicitly; provisioning polling is bounded.

SFT base and fine-tuned deployments use the SFT account; RFT base and fine-tuned deployments use the RFT account. Configure an existing o3-mini grader separately. To compare preprovisioned deployments, retain successful training state/configuration, run the saved-state loading and deployment-readiness cells, and then run the comparison cells. Skip the upload/training cells when the recorded jobs have already completed. No creation guard flag, capacity setting, or UUID suffix is required.

In [ ]:
def deployment_specifications():
    specs = []
    names = [required(name) for name in ['FOUNDRY_SFT_BASE_DEPLOYMENT', 'FOUNDRY_SFT_DEPLOYMENT', 'FOUNDRY_RFT_BASE_DEPLOYMENT', 'FOUNDRY_RFT_DEPLOYMENT']]
    if len(set(names)) != 4 or any(not re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,63}', name) for name in names):
        raise ValueError('Choose four distinct valid unique deployment names.')
    for index, (label, endpoint, model, version) in enumerate([
        ('sft', SFT_ENDPOINT, 'gpt-4.1-mini', '2025-04-14'),
        ('sft', SFT_ENDPOINT, state['jobs'].get('sft', {}).get('fine_tuned_model'), '1'),
        ('rft', RFT_ENDPOINT, 'o4-mini', '2025-04-16'),
        ('rft', RFT_ENDPOINT, state['jobs'].get('rft', {}).get('fine_tuned_model'), '1'),
    ]):
        if not model or state['jobs'].get(label, {}).get('status') != 'succeeded':
            raise RuntimeError(f'{label} requires a saved successful model artifact.')
        account = required(f'FOUNDRY_{label.upper()}_ACCOUNT_NAME')
        if urlparse(endpoint).hostname != f'{account}.services.ai.azure.com':
            raise ValueError(f'{label} account must match the configured Foundry endpoint.')
        specs.append((account, names[index], model, version, label))
    return specs

def arm_url(account, name):
    subscription = quote(required('FOUNDRY_SUBSCRIPTION_ID'), safe='')
    group = quote(required('FOUNDRY_RESOURCE_GROUP'), safe='')
    return f'https://management.azure.com/subscriptions/{subscription}/resourceGroups/{group}/providers/Microsoft.CognitiveServices/accounts/{quote(account, safe="")}/deployments/{quote(name, safe="")}?api-version=2024-10-01'

def arm_headers():
    return {'Authorization': f"Bearer {credential.get_token('https://management.azure.com/.default').token}", 'Content-Type': 'application/json'}

def verify_deployment(resource, model, version):
    actual = resource.get('properties', {}).get('model', {})
    expected = {'format': 'OpenAI', 'name': model, 'version': version}
    if any(actual.get(key) != value for key, value in expected.items()) or resource.get('sku', {}).get('name') != 'GlobalStandard':
        raise RuntimeError('Deployment name collision: model/version/format/SKU differs. Choose a new unique name; no resource was changed.')

def ensure_deployment(account, name, model, version):
    import requests
    url = arm_url(account, name)
    current = requests.get(url, headers=arm_headers(), timeout=60)
    if current.status_code == 404:
        raise RuntimeError(f'Missing deployment {name}; provision it through your approved process, then rerun discovery.')
    current.raise_for_status()
    verify_deployment(current.json(), model, version)
    deadline = time.monotonic() + 600
    while time.monotonic() < deadline:
        current = requests.get(url, headers=arm_headers(), timeout=60)
        current.raise_for_status()
        resource = current.json()
        verify_deployment(resource, model, version)
        status = resource['properties'].get('provisioningState')
        if status == 'Succeeded':
            return resource
        if status in {'Failed', 'Canceled', 'Cancelled', 'Deleting'}:
            raise RuntimeError(f'Deployment {name} ended in {status}. No cleanup operation is attempted.')
        time.sleep(15)
    raise TimeoutError(f'Deployment {name} did not become ready within 600 seconds.')

deployment_evidence = {}
specs = deployment_specifications()
required('FOUNDRY_SUBSCRIPTION_ID')
required('FOUNDRY_RESOURCE_GROUP')
for account, name, model, version, label in specs:
    deployment_evidence[name] = ensure_deployment(account, name, model, version)
(OUTPUTS / 'deployment-state.json').write_text(json.dumps(deployment_evidence, indent=2), encoding='utf-8')


## 8. Optional SFT invocation and base-versus-fine-tuned comparison

Both deployments receive all 280 held-out next-tool decisions with identical generation settings. Each row must expect exactly one call. Missing or extra predictions score zero; a single prediction is graded with the checked-in next-tool grader and is also measured for exact and tool-name accuracy. SFT predicts calls but never executes them.

Each completed deployment side is checkpointed immediately. A rerun reuses a side only when its project endpoint, deployment names, input bytes, grader bytes, row count, and generation settings match the saved comparison context. An interruption within one side repeats that incomplete side but does not repeat a completed side.

In [ ]:
grader_spec = importlib.util.spec_from_file_location('retail_next_tool_grader', ROOT / 'graders' / 'next-tool-call-grader.py')
next_tool_grader = importlib.util.module_from_spec(grader_spec)
grader_spec.loader.exec_module(next_tool_grader)

COMPARISON_CONTRACT_VERSION = 4

def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def deployment_context(deployment_names):
    context = {}
    for label, name in deployment_names.items():
        resource = deployment_evidence[name]
        model = resource.get('properties', {}).get('model', {})
        context[label] = {
            'name': name,
            'model': model.get('name'),
            'version': model.get('version'),
            'format': model.get('format'),
            'sku': resource.get('sku', {}).get('name'),
        }
    return context

def load_comparison_results(path, context):
    if not path.exists():
        return {}
    saved = json.loads(path.read_text(encoding='utf-8'))
    if saved.get('context') != context:
        print(f'Ignoring {path.name}: comparison context changed.')
        return {}
    results = saved.get('results', {})
    if not isinstance(results, dict):
        raise RuntimeError(f'{path.name} has an invalid results object.')
    return results

def save_comparison_result(path, context, label, result):
    results = load_comparison_results(path, context)
    results[label] = result
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps({'context': context, 'results': results}, indent=2) + '\n', encoding='utf-8')
    return results

def score_next_tool_row(calls, item):
    expected = item['expected_output']['tool_calls']
    if len(expected) != 1:
        raise ValueError('Next-tool comparison rows must expect exactly one call.')
    if len(calls) != 1:
        return {'reward': 0.0, 'exact': False, 'tool_name_match': False}
    return {'reward': next_tool_grader.grade({'output_tools': calls}, item),
        'exact': next_tool_grader._sim(calls[0], expected[0]) == 1.0,
        'tool_name_match': calls[0]['function']['name'] == expected[0]['function']['name']}

from azure.ai.projects.models import ModelDeployment
GRADER_ENDPOINT = project_endpoint('FOUNDRY_GRADER_PROJECT_ENDPOINT')
GRADER_DEPLOYMENT = required('FOUNDRY_RFT_GRADER_DEPLOYMENT')
grader_project = AIProjectClient(endpoint=GRADER_ENDPOINT, credential=credential, allow_preview=True)
resources_to_close.append(grader_project)
grader_resource = grader_project.deployments.get(GRADER_DEPLOYMENT)
if not isinstance(grader_resource, ModelDeployment) or grader_resource.model_name != 'o3-mini':
    raise RuntimeError('Configured grader deployment must serve the o3-mini model.')
grader_client = grader_project.get_openai_client(max_retries=0)
resources_to_close.append(grader_client)

def score_sft_deployment(deployment, rows):
    from concurrent.futures import ThreadPoolExecutor
    def score_one(row):
        response = retry_call(lambda: sft_client.chat.completions.create(model=deployment,
            messages=row['item']['messages'], tools=row['item']['tools'], tool_choice='required',
            temperature=0, max_completion_tokens=1024), policy=COMPARISON_RETRY_POLICY,
            operation_name='SFT comparison')
        if COMPARISON_PACING_SECONDS:
            time.sleep(COMPARISON_PACING_SECONDS)
        calls = [call.model_dump(exclude_none=True) for call in response.choices[0].message.tool_calls or []]
        return score_next_tool_row(calls, row['item'])
    if not rows:
        raise ValueError('Comparison rows must not be empty.')
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        results = list(pool.map(score_one, rows))
    return {'total': len(results), 'mean_single_call_grader_reward': sum(item['reward'] for item in results) / len(results),
        'exact_rate': sum(item['exact'] for item in results) / len(results),
        'tool_name_match_rate': sum(item['tool_name_match'] for item in results) / len(results)}

sft_checkpoint = OUTPUTS / 'sft-comparison-state.json'
sft_input_path = checked_prepared_path('next-tool-evaluation.jsonl')
sft_input_bytes = checked_prepared_bytes('next-tool-evaluation.jsonl')
sft_eval_rows = [json.loads(line) for line in sft_input_bytes.splitlines()]
sft_deployments = {label: required(name) for label, name in [
    ('base', 'FOUNDRY_SFT_BASE_DEPLOYMENT'), ('fine_tuned', 'FOUNDRY_SFT_DEPLOYMENT')]}
sft_context = {
    'contract_version': COMPARISON_CONTRACT_VERSION,
    'kind': 'sft-next-tool',
    'project_endpoint': SFT_ENDPOINT,
    'deployments': deployment_context(sft_deployments),
    'input_path': sft_input_path.relative_to(ROOT).as_posix(),
    'input_sha256': hashlib.sha256(sft_input_bytes).hexdigest(),
    'preparation_manifest_sha256': validated_inputs['outputs/prepared-data/manifest.json'],
    'grader_sha256': file_sha256(ROOT / 'graders' / 'next-tool-call-grader.py'),
    'row_count': len(sft_eval_rows),
    'generation': {'tool_choice': 'required', 'temperature': 0, 'max_completion_tokens': 1024},
}
sft_comparison = load_comparison_results(sft_checkpoint, sft_context)
warn_paid_once()
for label, deployment in sft_deployments.items():
    if label not in sft_comparison:
        result = score_sft_deployment(deployment, sft_eval_rows)
        sft_comparison = save_comparison_result(
            sft_checkpoint, sft_context, label, result
        )
print(sft_comparison)

## 9. Optional read-only RFT invocation and policy comparison

The comparison excludes scenario 7 and validates that each remaining reference sequence uses only the read-only allowlist. Every model-generated tool call is checked before execution. If any call in a batch is unknown or mutating, no call in that batch is executed and the scenario receives a deterministic score of zero without invoking the policy grader.

The base and fine-tuned deployments use the same nine scenarios, generation settings, external tool server, and checked-in policy grader. Authenticated read-only preflight runs before model invocation. Each completed side is checkpointed immediately; the comparison context includes the tool URL, active API hash, and auth fingerprint, never the token. A different server or credential invalidates reuse. An interruption within one side repeats that incomplete side but does not repeat a completed side.

In [ ]:
READ_ONLY_TOOLS = frozenset({'calculate', 'find_user_id_by_email', 'find_user_id_by_name_zip',
    'get_order_details', 'get_product_details', 'get_user_details', 'list_all_product_types',
    'list_user_orders', 'policy_verify_return'})

def execute_remote_tool(call, *, trace_id):
    if call.function.name not in READ_ONLY_TOOLS:
        raise RuntimeError(f'Blocked mutating or unknown tool: {call.function.name}')
    if not env_flag('FOUNDRY_TOOL_SERVER_IS_SANDBOX'):
        raise RuntimeError('Remote tools require an acknowledged isolated sandbox.')
    return retry_call(lambda: call_read_only_tool(
        TOOL_SERVER_URL, call.function.name, call.function.arguments, call.id,
        trace_id=trace_id, headers=TOOL_HEADERS, sandbox=True), operation_name='read-only retail tool')

def invoke_rft(client, deployment, row):
    trace_id = f'retail-{uuid.uuid4().hex}'
    messages, attempted_tools, executed_tools = list(row['messages']), [], []
    for _ in range(12):
        response = retry_call(lambda: client.chat.completions.create(model=deployment, messages=messages,
            tools=row['tools'], tool_choice='auto', max_completion_tokens=4096, reasoning_effort='medium'),
            policy=COMPARISON_RETRY_POLICY, operation_name='RFT invocation')
        if COMPARISON_PACING_SECONDS:
            time.sleep(COMPARISON_PACING_SECONDS)
        message = response.choices[0].message
        if not message.tool_calls:
            if not message.content:
                raise RuntimeError('RFT returned no final answer or tool calls.')
            return {'output': message.content, 'attempted_tools': attempted_tools,
                'executed_tools': executed_tools, 'blocked_tools': []}
        batch_names = [call.function.name for call in message.tool_calls]
        attempted_tools.extend(batch_names)
        blocked_tools = [name for name in batch_names if name not in READ_ONLY_TOOLS]
        if blocked_tools:
            return {'output': None, 'attempted_tools': attempted_tools,
                'executed_tools': executed_tools, 'blocked_tools': blocked_tools}
        messages.append({'role': 'assistant', 'content': message.content,
            'tool_calls': [call.model_dump(exclude_none=True) for call in message.tool_calls]})
        for call in message.tool_calls:
            messages.append({'role': 'tool', 'tool_call_id': call.id,
                'content': execute_remote_tool(call, trace_id=trace_id)})
            executed_tools.append(call.function.name)
    raise RuntimeError('RFT invocation exceeded 12 model turns.')

def parse_policy_score(content):
    text = (content or '').strip()
    if not re.fullmatch(r'(?:0(?:\.\d+)?|1(?:\.0+)?)', text):
        raise RuntimeError('Policy grader must return only a numeric score in [0, 1].')
    return float(text)

def policy_score(row, output):
    replacements = {'{{item.reference_policy_args}}': json.dumps(row['reference_policy_args']),
        '{{item.reference_policy_outcome}}': json.dumps(row['reference_policy_outcome']), '{{sample.output_text}}': output}
    messages = [{'role': item['role'], 'content': re.sub(r'\{\{(?:item\.(?:reference_policy_args|reference_policy_outcome)|sample\.output_text)\}\}',
        lambda match: replacements[match.group(0)], item['content'])} for item in policy_grader['input']]
    response = retry_call(lambda: grader_client.chat.completions.create(model=GRADER_DEPLOYMENT, messages=messages,
        max_completion_tokens=4096, reasoning_effort='low'),
        policy=COMPARISON_RETRY_POLICY, operation_name='policy grading')
    if COMPARISON_PACING_SECONDS:
        time.sleep(COMPARISON_PACING_SECONDS)
    return parse_policy_score(response.choices[0].message.content)

def score_rft_outcome(row, outcome):
    blocked = bool(outcome['blocked_tools'])
    score = 0.0 if blocked else policy_score(row, outcome['output'])
    return {
        'score': score,
        'status': 'blocked' if blocked else 'graded',
        'attempted_tools': outcome['attempted_tools'],
        'executed_tools': outcome['executed_tools'],
        'blocked_tools': outcome['blocked_tools'],
        'reference_tools': row['reference_tool_calls'],
    }

def select_read_only_scenarios(rows):
    selected = [(index, row) for index, row in enumerate(rows) if index != 7]
    for index, row in selected:
        if any(name not in READ_ONLY_TOOLS for name in row['reference_tool_calls']):
            raise RuntimeError(f'Scenario {index} has a mutating or unknown reference call.')
    if len(selected) != 9:
        raise RuntimeError('Expected nine read-only comparison scenarios.')
    return selected

def score_rft_deployment(client, deployment, rows):
    results = []
    for index, row in rows:
        result = score_rft_outcome(row, invoke_rft(client, deployment, row))
        result['index'] = index
        results.append(result)
    return {'mean_policy_reward': sum(item['score'] for item in results) / len(results),
        'pass_count': sum(item['status'] == 'graded' and item['score'] >= policy_grader['pass_threshold'] for item in results),
        'tool_sequence_exact': sum(item['status'] == 'graded' and item['executed_tools'] == item['reference_tools'] for item in results),
        'blocked_count': sum(item['status'] == 'blocked' for item in results),
        'total': len(results), 'results': results}

def rft_tool_equivalence():
    return {'server_url': TOOL_SERVER_URL, 'auth_fingerprint': tool_auth_fingerprint(),
        'openapi_sha256': file_sha256(CONFIGS / 'retail-openapi-with-policy.json'),
        'trace_scope': 'per-conversation-v1'}

rft_input_path = checked_prepared_path('rft-validation.jsonl')
rft_input_bytes = checked_prepared_bytes('rft-validation.jsonl')
rft_rows = [json.loads(line) for line in rft_input_bytes.splitlines()]
rft_selected = select_read_only_scenarios(rft_rows)
rft_checkpoint = OUTPUTS / 'rft-comparison-state.json'
rft_deployments = {label: required(name) for label, name in [
    ('base', 'FOUNDRY_RFT_BASE_DEPLOYMENT'), ('fine_tuned', 'FOUNDRY_RFT_DEPLOYMENT')]}
rft_context = {
    'contract_version': COMPARISON_CONTRACT_VERSION,
    'kind': 'rft-policy',
    'project_endpoint': RFT_ENDPOINT,
    'grader_project_endpoint': GRADER_ENDPOINT,
    'grader_deployment': {
        'name': GRADER_DEPLOYMENT,
        'model': grader_resource.model_name,
        'version': getattr(grader_resource, 'model_version', None),
    },
    'deployments': deployment_context(rft_deployments),
    'input_path': rft_input_path.relative_to(ROOT).as_posix(),
    'input_sha256': hashlib.sha256(rft_input_bytes).hexdigest(),
    'preparation_manifest_sha256': validated_inputs['outputs/prepared-data/manifest.json'],
    'grader_sha256': file_sha256(CONFIGS / 'rft-policy-grader.json'),
    'selected_indices': [index for index, _ in rft_selected],
    'read_only_tools': sorted(READ_ONLY_TOOLS),
    'tool_equivalence': rft_tool_equivalence(),
    'generation': {'tool_choice': 'auto', 'max_completion_tokens': 4096, 'reasoning_effort': 'medium'},
    'policy_grading': {'max_completion_tokens': 4096, 'reasoning_effort': 'low'},
}
rft_comparison = load_comparison_results(rft_checkpoint, rft_context)
warn_paid_once()
preflight_retail_tools(ROOT, TOOL_SERVER_URL, dataset_paths=prepared_paths, headers=TOOL_HEADERS,
    sandbox=env_flag('FOUNDRY_TOOL_SERVER_IS_SANDBOX'))
for label, deployment in rft_deployments.items():
    if label not in rft_comparison:
        result = score_rft_deployment(rft_client, deployment, rft_selected)
        rft_comparison = save_comparison_result(
            rft_checkpoint, rft_context, label, result
        )
(OUTPUTS / 'comparison-state.json').write_text(
    json.dumps({'sft': {'context': sft_context, 'results': sft_comparison},
        'rft': {'context': rft_context, 'results': rft_comparison}}, indent=2) + '\n',
    encoding='utf-8')
print(rft_comparison)

## 10. Interpret the results

Training loss and reward are service telemetry, not held-out customer quality. Compare the base and fine-tuned deployments only when their saved context matches the current inputs, graders, models, and generation settings. Fine-tuning is not guaranteed to improve either comparison; retain regressions and blocked outcomes in your conclusions.

The comparison state under `outputs/` can contain customer resource names and service results. Keep it private and remove it when it is no longer required.

Run the final cell to close the clients created by this notebook. Closing clients does not cancel jobs, unload deployments, or delete resources.

In [ ]:
for resource in reversed(resources_to_close):
    resource.close()
resources_to_close.clear()